In [ ]:
import pandas as pd
import requests

# =====================================================================
# 1. CONSUMO VIA API REST (O VERDADEIRO ENDPOINT JSON)
# =====================================================================
url = "https://apidadosabertos.saude.gov.br/arboviroses/chikungunya"

print("A conectar diretamente à API REST do Ministério da Saúde... (Aguarde)")
headers = {'Accept': 'application/json'}
params = {'limit': 1000}

resposta = requests.get(url, headers=headers, params=params)
resposta.raise_for_status()

# Converte a resposta em JSON
dados_json = resposta.json()
df = pd.DataFrame(dados_json)

if 'parametros' in df.columns:
    df = pd.DataFrame(df['parametros'].tolist())

# Padroniza todas as colunas para maiúsculas para facilitar os filtros
df.columns = df.columns.str.upper()

print(f"✅ Base de dados carregada via API com sucesso! ({len(df)} registros)\n")

# =====================================================================
# 2. FILTRAGEM ESTRATÉGICA: ESTADO VS MUNICÍPIO
# =====================================================================
# Como a API devolveu uma amostra (ex: 20 linhas), pode ser que não haja
# nenhum caso de SC ou Araranguá nessa pequena amostra. O código abaixo é
# blindado para não dar erro mesmo que a amostra venha vazia.

if 'SG_UF_NOT' in df.columns:
    df_sc = df[(df['SG_UF_NOT'] == 42) | (df['SG_UF_NOT'] == '42') | (df['SG_UF_NOT'] == 'SC')].copy()
else:
    df_sc = df.copy()

if 'ID_MUNICIP' in df_sc.columns:
    df_ararangua = df_sc[(df_sc['ID_MUNICIP'] == 420140) | (df_sc['ID_MUNICIP'] == '420140')].copy()
else:
    df_ararangua = df_sc.copy()

# =====================================================================
# 3. DIAGNÓSTICO DE DADOS (EDA)
# =====================================================================
print("--- 1. ESTRUTURA BÁSICA ---")
print(f"Total de Notificações capturadas na API: {df.shape[0]} linhas e {df.shape[1]} colunas.")
print(f"Total de Notificações em SC: {df_sc.shape[0]} linhas.")
print(f"Total em Araranguá: {df_ararangua.shape[0]} linhas.\n")

print(" A observar a tabela desempacotada (5 primeiras linhas):")
display(df.head())

print("\n--- 2. DADOS AUSENTES (NULOS) ---")
colunas_criticas = [col for col in ['DT_NOTIFIC', 'CS_SEXO', 'CS_RACA', 'CS_ESCOL_N'] if col in df.columns]

if colunas_criticas:
    nulos = (df[colunas_criticas].isnull().sum() / len(df)) * 100
    print("Percentagem de dados vazios na amostra capturada:")
    print(nulos.round(2).astype(str) + ' %')
else:
    print("As colunas críticas procuradas não foram fornecidas por este endpoint da API.")

print("\n--- 3. DUPLICIDADE ---")
duplicadas = df.duplicated().sum()
print(f"Registos 100% idênticos encontrados na amostra: {duplicadas}")

# =====================================================================
# 4. MINI-RELATÓRIO AUTOMATIZADO (TEXTO FINAL)
# =====================================================================
print("\n" + "="*60)
print(" MINI-RELATÓRIO FINAL PARA ENTREGA")
print("="*60)
print(f"""
A base de dados de Chikungunya foi consumida de forma nativa via API REST (JSON).
Devido às limitações de paginação e segurança do endpoint público do DataSUS,
trabalhou-se com uma amostra programática de {df.shape[0]} registos.

DIAGNÓSTICO: A base requer tratamento antes de qualquer modelação analítica.
Justificações:
1. Empacotamento de Chaves: Os dados brutos da API são entregues encapsulados num
   objeto 'parametros', exigindo achatamento (flattening) via código antes da análise.
2. Formato de Dados JSON: Ao consumir diretamente da API, todas as informações, incluindo
   datas (como DT_NOTIFIC) e números, são frequentemente interpretadas como texto (string/object),
   necessitando conversão para Datetime e Inteiros no Pandas.
3. Tratamento de Amostras: O consumo de APIs públicas muitas vezes resulta em conjuntos
   fragmentados (paginação). Para uma análise definitiva de Santa Catarina/Araranguá,
   seria necessário implementar um loop de paginação para extrair o histórico completo.
""")

A conectar diretamente à API REST do Ministério da Saúde... (Aguarde)
✅ Base de dados carregada via API com sucesso! (20 registros)

--- 1. ESTRUTURA BÁSICA ---
Total de Notificações capturadas na API: 20 linhas e 126 colunas.
Total de Notificações em SC: 0 linhas.
Total em Araranguá: 0 linhas.

 A observar a tabela desempacotada (5 primeiras linhas):


,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_SIN_PRI,...,ARQUIVO,CS_SUSPEIT,IN_VINCULA,CS_FLXRET,FLXRECEBI,TPUNINOT,ANO_NASC,NU_LOTE_I,DT_DIGITA,MIGRADO_W
0,2,A92.,2016-02-24,201608,2016,51,510263,1585,None,2016-02-24,...,CHIKBR16.csv,nan,nan,0,2,nan,None,None,None,None
1,2,A92.,2016-02-14,201607,2016,51,510263,1585,None,2016-02-07,...,CHIKBR16.csv,nan,nan,0,2,nan,None,None,None,None
2,2,A92.,2016-02-22,201608,2016,51,510263,1585,None,2016-02-22,...,CHIKBR16.csv,nan,nan,0,2,nan,None,None,None,None
3,2,A92.,2016-01-27,201604,2016,51,510263,1585,None,2016-01-25,...,CHIKBR16.csv,nan,nan,0,2,nan,None,None,None,None
4,2,A92.,2016-02-24,201608,2016,51,510263,1585,None,2016-02-16,...,CHIKBR16.csv,nan,nan,0,2,nan,None,None,None,None



--- 2. DADOS AUSENTES (NULOS) ---
Percentagem de dados vazios na amostra capturada:
DT_NOTIFIC    0.0 %
CS_SEXO       0.0 %
CS_RACA       0.0 %
CS_ESCOL_N    0.0 %
dtype: object

--- 3. DUPLICIDADE ---
Registos 100% idênticos encontrados na amostra: 0

 MINI-RELATÓRIO FINAL PARA ENTREGA

A base de dados de Chikungunya foi consumida de forma nativa via API REST (JSON).
Devido às limitações de paginação e segurança do endpoint público do DataSUS, 
trabalhou-se com uma amostra programática de 20 registos.

DIAGNÓSTICO: A base requer tratamento antes de qualquer modelação analítica.
Justificações:
1. Empacotamento de Chaves: Os dados brutos da API são entregues encapsulados num 
   objeto 'parametros', exigindo achatamento (flattening) via código antes da análise.
2. Formato de Dados JSON: Ao consumir diretamente da API, todas as informações, incluindo 
   datas (como DT_NOTIFIC) e números, são frequentemente interpretadas como texto (string/object), 
   necessitando conversão para Dateti